In [1]:
%run 0_1_load_paths.ipynb

In [2]:
import os
import subprocess

import credentials
import momapy_kb.lpg.backends.neo4j
import momapy_kb.lpg.session

In [3]:
backend = momapy_kb.lpg.backends.neo4j.Neo4jBackend(
    hostname=credentials.NEO4J_URI,
    username=credentials.NEO4J_USERNAME,
    password=credentials.NEO4J_PASSWORD,
    notifications_min_severity="off",
)
session = momapy_kb.lpg.session.Session(backend)

<div class="alert alert-danger">We delete all the data from the DB</div>

In [4]:
session.delete_all()

## Adding AD BEL KG to the Neo4j database

We import the cypher dump:

In [5]:
command = [
    "cat",
    AD_KG_CYPHER_FILE,
    "|",
    "cypher-shell",
    "-a",
    credentials.NEO4J_URI,
    "-u",
    credentials.NEO4J_USERNAME,
    "-p",
    credentials.NEO4J_PASSWORD,
    "-d",
    credentials.NEO4J_DATABASE,
]

In [6]:
subprocess.run(" ".join(command), shell=True)

CompletedProcess(args='cat ../../data/ad_kg/cypher/ad_kg.cypher | cypher-shell -a localhost -u neo4j -p neofourj -d neo4j', returncode=0)

We make the Collection, Entry and Model nodes:

In [7]:
query = f"""
    MERGE
        (collection:Collection {{name: 'AD_KG_BEL'}})-[:HAS_ENTRY]->(collection_entry:CollectionEntry {{file_path: '{AD_KG_CYPHER_FILE}'}})-[:HAS_OBJ]->(model:BELModel)
    RETURN
        collection, collection_entry, model
"""
_ = session.execute_query(query)

In [8]:
query = """
    MATCH (n)
    WHERE NOT n:Collection AND NOT n:CollectionEntry AND NOT n:BELModel
    SET n:BELModelElement
    RETURN n
"""
_ = session.execute_query(query)

We link each node of the dump to the newly created model:

In [9]:
query = """
    MATCH (model_element:BELModelElement), (model:BELModel)
    MERGE (model)-[:HAS_NODE]->(model_element)
    RETURN model, model_element
"""
_ = session.execute_query(query)

We extract subgraph information from relationships and make Subgraph nodes

In [10]:
query = """
    MATCH (n)-[r]->(m), (model:BELModel)
    UNWIND r.annotationSubgraph AS subgraph
    MERGE (model)-[:HAS_SUBGRAPH]->(subgraph_node:Subgraph {name: subgraph})
    RETURN subgraph_node
"""
_ = session.execute_query(query)

We add nodes to subgraphs:

In [11]:
query = """
CALL () {
    MATCH (n)-[r]->(m)
    UNWIND r.annotationSubgraph AS subgraph
    RETURN n AS n, subgraph AS subgraph
    UNION
    MATCH (m)-[r]->(n)
    UNWIND r.annotationSubgraph AS subgraph
    RETURN n AS n, subgraph AS subgraph
}
MATCH (subgraph_node:Subgraph)
WHERE subgraph_node.name = subgraph
MERGE (subgraph_node)-[:HAS_NODE]->(n)
RETURN subgraph_node, n
"""
_ = session.execute_query(query)

We make a special Subgraph node with name "main_model" for nodes which do not belong to a subgraph (in order to have uniform queries later):

In [12]:
query = """
    MATCH (model:BELModel)
    MERGE (model)-[:HAS_SUBGRAPH]->(subgraph_node:Subgraph {name: 'main_model'})
    RETURN subgraph_node
"""
_ = session.execute_query(query)

We add all nodes that do not belong to a subgraph to the "main_model" Subgraph node:

In [13]:
query = """
    MATCH (n:BELModelElement), (subgraph_node:Subgraph {name: 'main_model'})
    WHERE NOT EXISTS {(n)<-[r:HAS_NODE]-(s:Subgraph)}
    MERGE (subgraph_node)-[:HAS_NODE]->(n)
    RETURN n
"""
_ = session.execute_query(query)

## Adding the COVID-19 DM and PD DM to the Neo4j database

We save the collection to the DB:

In [14]:
collection_names_and_input_file_paths = [
    (
        "COVID_DM_CD",
        COVID_DM_CD_DATA_DIR.glob("*.xml"),
    ),
    (
        "PD_DM_CD",
        PD_DM_CD_DATA_DIR.glob("*.xml"),
    ),
]

In [15]:
session.save_collections_from_file_paths(
    collection_names_and_input_file_paths,
    return_type="map",
    with_membership_edges=True,
    integration_mode="hash",
)

/home/rougny/code/momapy/src/momapy/celldesigner/io/celldesigner/reader.py:795: UserWarning: skipping modulation 'ir3e6': references a Degraded species (source alias='csa163', target alias='ir15b'); Degraded species have no model peer.
  cls._make_and_add_modulation(
